# 1 · Make delphinID detection frames

This notebook turns **PAMGuard detections** plus **your annotations** into a **detection frames CSV**, the training data for `02_train_delphinID`.

**You need**
1. A folder of PAMGuard binary files (`.pgdf`) from the Click Detector or the Whistle and Moan Detector. Subfolders are searched too.
2. An annotations CSV: one row per labelled time period.

| column | meaning | example |
|---|---|---|
| `event_id` | encounter/event the period belongs to (rows can share one) | `enc001` |
| `label` | class, e.g. species code | `Dde` |
| `start` | period start, UTC | `2018-09-09 21:55:00` |
| `end` | period end, UTC | `2018-09-09 22:05:00` |
| *any other* | optional; copied to the frames (e.g. `location`) | `Site A` |

See `examples/annotations_example.csv`.

**You get** `<name>.csv` (one row per frame) and `<name>_params.json` (settings, used when exporting the model to PAMGuard).

A frame summarises the detections in a short window (default 4 s) as one spectrum, computed exactly as PAMGuard's delphinID module does. Frames are made within each event, never across events.

## Setup
In Google Colab, run the next cell to install the package. Locally, install once with `pip install -e "python[pamguard]"` from the repository folder and skip it.

In [ ]:
# Google Colab only
# %pip install -q "speciesclassifiers[pamguard] @ git+https://github.com/tristankleyn/speciesClassifiers#subdirectory=python"

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from speciesclassifiers.delphinid import AcousticParams, make_frames

## Parameters
Edit these, then run all cells.

The defaults match the published delphinID models. The number of model inputs follows from the frequency range, FFT length, sample rate (clicks) and downsampling; it's printed below.

In [ ]:
# --- Data ----------------------------------------------------------------------------------
BINARY_FOLDER   = "path/to/binaries"            # folder of PAMGuard .pgdf files
ANNOTATIONS_CSV = "path/to/annotations.csv"     # see the table above
VOC_TYPE        = "whistle"                     # "whistle" or "click"
OUTPUT_NAME     = "frames_whistles"             # writes OUTPUT_NAME.csv and OUTPUT_NAME_params.json
SAMPLE_RATE     = None                          # Hz; None = work it out from the binaries

# --- Acoustic parameters -------------------------------------------------------------------
FREQ_RANGE  = (2000, 20000)   # Hz. Published models: whistles (2000, 20000), clicks (10000, 40000)
FRAME_LEN   = 4.0             # frame length, s
HOP         = 4.0             # step between frame starts, s (smaller than FRAME_LEN = overlapping frames)
DOWNSAMPLE  = 2               # average this many neighbouring bins

# whistles only
MIN_DENSITY = 0.05            # minimum whistle density (contour points / time slices) in a frame
MIN_FRAG_MS = 200             # ignore contours shorter than this (ms)

# clicks only
MIN_CLICKS    = 3             # minimum clicks in a frame
FFT_LEN       = 512           # FFT length for click spectra
SMOOTH_WINDOW = 3             # moving-average window (bins)

In [ ]:
params = AcousticParams(
    voc_type=VOC_TYPE, freq_range=FREQ_RANGE, frame_len=FRAME_LEN, hop=HOP, downsample=DOWNSAMPLE,
    min_density=MIN_DENSITY, min_frag_ms=MIN_FRAG_MS,
    min_clicks=MIN_CLICKS, fft_len=FFT_LEN, smooth_window=SMOOTH_WINDOW,
)
params

## Make frames
Only binary files that overlap an annotated period are read.

In [ ]:
frames, info = make_frames(BINARY_FOLDER, ANNOTATIONS_CSV, params, sample_rate=SAMPLE_RATE)
sample_rate = info["sample_rate"]
print("Model inputs per frame:", params.n_bins(sample_rate))
frames.head()

## Check
Frames per label and per event. Labels with few events will be hard to classify reliably: cross-validation needs several independent events per label.

In [ ]:
summary = frames.groupby("label").agg(events=("event_id", "nunique"), frames=("event_id", "size"))
summary["frames_per_event"] = (summary["frames"] / summary["events"]).round(1)
summary

In [ ]:
frames.groupby(["label", "event_id"]).size().rename("frames").to_frame().T

Median frame per label, shaded between the 25th and 75th percentiles. Labels the model can tell apart usually differ here.

In [ ]:
feat = [c for c in frames.columns if c[0] == "f" and c[1:].isdigit()]
freqs = np.linspace(*FREQ_RANGE, len(feat) + 1)[:-1] + (FREQ_RANGE[1] - FREQ_RANGE[0]) / len(feat) / 2
fig, ax = plt.subplots(figsize=(8, 4))
for lab, g in frames.groupby("label"):
    q = g[feat].quantile([0.25, 0.5, 0.75]).to_numpy()
    line, = ax.plot(freqs / 1000, q[1], label=f"{lab} (n={len(g)})")
    ax.fill_between(freqs / 1000, q[0], q[2], alpha=0.15, color=line.get_color())
ax.set(xlabel="Frequency (kHz)", ylabel="Normalised energy" if VOC_TYPE == "click" else "Share of contour points",
       title=f"Median {VOC_TYPE} frame per label")
ax.legend()
plt.show()

## Save

In [ ]:
out_csv = Path(f"{OUTPUT_NAME}.csv")
frames.to_csv(out_csv, index=False)
meta = {"acoustic_params": params.to_dict(), "sample_rate": sample_rate, "recording_settings": info,
        "binary_folder": str(BINARY_FOLDER), "annotations_csv": str(ANNOTATIONS_CSV)}
Path(f"{OUTPUT_NAME}_params.json").write_text(json.dumps(meta, indent=1, default=str))
print(f"Saved {len(frames)} frames to {out_csv.resolve()}")

## Frames CSV format
If you make frames another way, `02_train_delphinID` accepts any CSV with these columns:

| column | meaning |
|---|---|
| `event_id` | event the frame belongs to |
| `label` | class |
| `frame_start` | frame start time (Unix seconds) |
| `n_detections` | detections in the frame |
| `density` | whistles only: whistle density |
| other columns | copied from the annotations (e.g. `location`) |
| `f1` … `fN` | the frame's spectrum (sums to 1) |

For a model to work in PAMGuard, the features must be computed as here (same transforms and parameters).